In [29]:
import requests
import pandas as pd
import numpy as np
from keys import OWM_key  # Tu API key de OpenWeatherMap
import sqlalchemy
import pymysql




In [30]:
city = "Busan"
country = "KR"

url = f"http://api.openweathermap.org/data/2.5/forecast?q={city},{country}&appid={OWM_key}&units=metric&lang=en"
response = requests.get(url)

if response.status_code == 200:
    data = response.json()
    print("API OK")
else:
    print("Error en la petición:", response.status_code)



API OK


In [31]:

forecast_list = data.get('list', [])

times = []
temperatures = []
humidities = []
weather_statuses = []
wind_speeds = []
rain_volumes = []
snow_volumes = []

for entry in forecast_list:
    times.append(entry.get('dt_txt', np.nan))
    temperatures.append(entry.get('main', {}).get('temp', np.nan))
    humidities.append(entry.get('main', {}).get('humidity', np.nan))
    weather_statuses.append(entry.get('weather', [{}])[0].get('main', np.nan))
    wind_speeds.append(entry.get('wind', {}).get('speed', np.nan))
    rain_volumes.append(entry.get('rain', {}).get('3h', np.nan))
    snow_volumes.append(entry.get('snow', {}).get('3h', np.nan))


In [43]:
municipality_iso_country = "Busan,KR"

df = pd.DataFrame({
    'weather_datetime': pd.to_datetime(times),
    'temperature': pd.to_numeric(temperatures, errors='coerce'),
    'humidity': pd.to_numeric(humidities, errors='coerce'),
    'weather_status': weather_statuses,
    'wind': pd.to_numeric(wind_speeds, errors='coerce'),
    'rain_qty': pd.to_numeric(rain_volumes, errors='coerce'),
    'snow': pd.to_numeric(snow_volumes, errors='coerce'),
    'municipality_iso_country': [municipality_iso_country] * len(times)  # ← importante
})

df.head()




,weather_datetime,temperature,humidity,weather_status,wind,rain_qty,snow,municipality_iso_country
0,2025-10-31 00:00:00,11.80,77,Rain,3.42,0.14,NaN,"Busan,KR"
1,2025-10-31 03:00:00,14.30,71,Rain,4.66,0.28,NaN,"Busan,KR"
2,2025-10-31 06:00:00,17.91,55,Clouds,3.29,NaN,NaN,"Busan,KR"
3,2025-10-31 09:00:00,17.34,56,Clear,2.69,NaN,NaN,"Busan,KR"
4,2025-10-31 12:00:00,15.98,65,Clear,2.31,NaN,NaN,"Busan,KR"


In [1]:
import sqlalchemy
import pymysql

schema = "gans"
host = "127.0.0.1"
user = "root"
password = "adricaba1107"
port = 3306

con = sqlalchemy.create_engine(f'mysql+pymysql://{user}:{password}@{host}:{port}/{schema}')



In [45]:
df.to_sql('weather_data', if_exists = 'append', con = con, index=False)

40